In [176]:
# import os
import json
import cv2
import shutil
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
# from pthPontos.MyNetwork import getModel
import random
# Caminho absoluto para a pasta 'src' (ajuste conforme sua estrutura)
src_path = os.path.abspath(os.path.join(os.getcwd(), 'src'))

# Adiciona ao sys.path
sys.path.append(src_path)

from datasets.weedsgalore.weedsgalore import WeedsGaloreDataset
from nets import deeplabv3plus_resnet50
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassCalibrationError, MulticlassConfusionMatrix

json_path = r'via_export_json.json'
root_images = r'imagens'
leitura = 'r'
imagens_path = r'data/nao_temporal/images'
semantic_path = r'data/nao_temporal/semantic'


In [177]:
def makeMasc(regions,semantic_path,size_mask = (600,600)):
    fundo = np.zeros(size_mask, dtype=np.uint8)
    for region in regions:
        tipo = region["region_attributes"]['type']
        xPoints = region["shape_attributes"]["all_points_x"]
        yPoints = region["shape_attributes"]["all_points_y"]
        points = np.column_stack((xPoints,yPoints))
        if tipo == "Milho":
            cv2.fillPoly(fundo,[points],1)
        elif tipo == "Daninha":
            cv2.fillPoly(fundo,[points],2)
    print(semantic_path)
    # plt.imsave(semantic_path,fundo,cmap='gray',vmin=0, vmax=2)
    cv2.imwrite(semantic_path, fundo)
    print('>>>>>',np.unique(fundo))

In [178]:
# images_names = [paths for paths in os.listdir(root_images) if paths.endswith('png')]
# # print(images_names)
# with open(json_path,leitura) as arquivo:
#     json_content = json.load(arquivo)
#     images_data = [value for key,value in json_content.items() if value["regions"]] 
#     # print(len(images_data))
#     if not os.path.exists(imagens_path):
#         os.makedirs(imagens_path)
#     if not os.path.exists(semantic_path):
#         os.makedirs(semantic_path)
#     for image in images_data:
#         image_name = image["filename"]
#         regions = image['regions']
#         image_origin = os.path.join(root_images,image_name)
#         image_destini = os.path.join(imagens_path,image_name)
#         semantic_destini = os.path.join(semantic_path,image_name)
#         shutil.copy(image_origin,image_destini)
#         makeMasc(regions,semantic_destini)


In [179]:
images_names = [paths for paths in os.listdir(root_images) if paths.endswith('png')]
# # print(images_names)
# with open(json_path,leitura) as arquivo:
#     json_content = json.load(arquivo)
#     images_data = [value for key,value in json_content.items() if value["regions"]] 
#     # print(len(images_data))
#     if not os.path.exists(imagens_path):
#         os.makedirs(imagens_path)
#     if not os.path.exists(semantic_path):
#         os.makedirs(semantic_path)
#     image = images_data[0]
#     image_name = image["filename"]
#     regions = image['regions']
#     image_origin = os.path.join(root_images,image_name)
#     image_destini = os.path.join(imagens_path,image_name)
#     semantic_destini = os.path.join(semantic_path,image_name)
#     shutil.copy(image_origin,image_destini)
#     makeMasc(regions,semantic_destini)

In [180]:
# sematins = os.listdir(semantic_path)
# for semantic in sematins:
#     atual_semantic = os.path.join(semantic_path,semantic)
#     atual_im = cv2.imread(atual_semantic,cv2.IMREAD_GRAYSCALE)
#     print(np.unique(atual_im))

In [181]:
def get_lenData():
    return len(os.listdir(semantic_path))
print(get_lenData())

74


In [182]:
dataset_path = r'data\nao_temporal'
dataset_size = get_lenData()
# dataset_size = None
in_bands = False
num_classes = 3
is_training = True
split = 'train'
augmentation = True
dataset = WeedsGaloreDataset( dataset_path, dataset_size, in_bands, num_classes, is_training, split, augmentation)
print(dataset.__len__())


74


In [183]:
modelo_inicial = r'pthPontos\dlv3p_rgb_3.pth'
def actual_model():
    modelos = [modelos for modelos in os.listdir() if modelos.endswith('pth')]
    sem_outro = len(modelos)<2
    return modelo_inicial if sem_outro else max(modelos)
def proximoModelo():
    if actual_model() == modelo_inicial:
        return '0_modelo_treinado.pth'
    else:
        nova_vercao = int(actual_model().split('_')[0])+1
        return f'{nova_vercao}_modelo_treinado.pth'
print(actual_model())
print(proximoModelo())

pthPontos\dlv3p_rgb_3.pth
0_modelo_treinado.pth


In [184]:
def plotImagens(images, semantics, predicts):
    num_images = len(images)
    fig, axes = plt.subplots(num_images, 3, figsize=(9, 3 * num_images))
    
    if num_images == 1:
        axes = [axes]  # Garante que axes seja iterável mesmo para um único item
    
    for i in range(num_images):
        img = images[i].permute(1, 2, 0).cpu().numpy()  # Converte tensor para imagem
        label = semantics[i].cpu().numpy()
        predict = torch.argmax(predicts[i], dim=0).cpu().numpy()
        
        axes[i][0].imshow(img)
        axes[i][0].set_title("Imagem")
        axes[i][0].axis("off")
        
        axes[i][1].imshow(label, cmap='gray')
        axes[i][1].set_title("Rótulo")
        axes[i][1].axis("off")
        
        axes[i][2].imshow(predict, cmap='gray')
        axes[i][2].set_title("Previsão")
        axes[i][2].axis("off")
    
    plt.tight_layout()
    plt.show()

In [185]:
num_classes = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device in use: {device.type}')
model = deeplabv3plus_resnet50(num_classes=num_classes)
model = model.to(device)

model.backbone.conv1 = nn.Conv2d(num_classes,model.backbone.conv1.out_channels,kernel_size=7,stride=2,padding=3,bias=False,device=device)

model_weights_dir = actual_model()
model_dict = torch.load(model_weights_dir,map_location=device)
model.load_state_dict(model_dict)

# dataset
dataloader = DataLoader(dataset=dataset,batch_size=1,shuffle=False,num_workers=1,collate_fn=None, drop_last=True)
data_iter = iter(dataloader)

evaluator = MulticlassJaccardIndex(num_classes=num_classes,average=None,ignore_index=None).to(device)

confmat = MulticlassConfusionMatrix(num_classes=num_classes, normalize='true', ignore_index=None).to(device)

model.eval()

Device in use: cpu


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [186]:

images = []
semantics = []
predicts = []
for idx in random.sample(range(get_lenData()),3):
    img, label, binary_label = dataset.__getitem__(idx)
    predict = model(img)
    images.append(img)
    semantics.append(label)
    predicts.append(predict)
plotImagens(images,semantics,predicts)

IndexError: index 59 is out of bounds for axis 0 with size 1

In [ ]:
def ajustarImages():
    rgb_path = 'data/rgb'
    rootatual = 'data/images'
    os.makedirs(rgb_path, exist_ok=True) 
    for image_path,image_name in [(os.path.join(rootatual,image_name),image_name) for image_name in os.listdir(rootatual) if image_name.endswith('png')]:
        image = cv2.imread(image_path,cv2.IMREAD_COLOR_RGB)
        r,g,b = cv2.split(image)
        print(cv2.imwrite(os.path.join(rgb_path,image_name.replace('image.','image_R.')),r))
        cv2.imwrite(os.path.join(rgb_path,image_name.replace('image.','image_G.')),g)
        cv2.imwrite(os.path.join(rgb_path,image_name.replace('image.','image_B.')),b)
# ajustarImages()

In [ ]:
# criterion = ...
# optimizer = ...

In [ ]:
# num_epochs = 3 

# for epoch in range(num_epochs):
#     model.train()
    
#     for batch in dataset:
#         inputs, targets = batch
#         outputs = model(inputs)
        
#         loss = criterion(outputs, targets)  # Supondo que criterion já está definido
#         optimizer.zero_grad()  # Supondo que optimizer já está definido
#         loss.backward()
#         optimizer.step()

#     print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {loss.item()}")

# # Salvar o modelo
# save_path = f"pthPontos/{proximoModelo()}"
# torch.save(model.state_dict(), save_path)
# print(f"Modelo salvo em {save_path}")
